# 00 · ProBayes Dataset — Exploratory Data Analysis
**Project** : Intelligent Multi-Sensor Quality Control for Injection Moulding
**Dataset** : ProBayes (SKZ + Fraunhofer IPA) — KraussMaffei 160-750PX, Verzugsschale (warpage shell)
**Author**  : Sandeep J S · 1MS24AD402 · M.S. Ramaiah Institute of Technology
**Guide**   : Dr. Sowmya B J | ADP67 Mini Project, Feb–May 2026

---
### What this notebook does
1. Dataset overview — shape, modalities, missing data
2. Label-imbalance audit (LBL_* columns)
3. **The Leakage Problem** — why random-split F1 is 1.00 but group-fold F1 is 0.30
4. Thermal image visualisation (Z01-Diff analysis)
5. CV image visualisation (3-section warpage shell)
6. DXP process-signal visualisation (8 key channels)
7. Taguchi DOE cell structure
8. Key findings & research motivation

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── locate fusion_system package ──────────────────────────────
NB_DIR   = Path.cwd()
FS_DIR   = NB_DIR.parent          # fusion_system/
sys.path.insert(0, str(FS_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

import config as C

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
print("Config OK  |  LABEL_COLS:", C.LABEL_COLS)
print("Paths:")
print("  THERMAL_CSV_DIR :", C.THERMAL_CSV_DIR)
print("  CV_IMAGE_DIR    :", C.CV_IMAGE_DIR)
print("  PARQUET_PATH    :", C.PARQUET_PATH)

## 1 · Load Dataset

In [ ]:
from data.loaders import load_parquet, build_cv_index, join_cv_to_parquet
from data.dataset import get_tabular_cols, get_dxp_cols

df = load_parquet(C.PARQUET_PATH)
print(f"Parquet loaded: {df.shape[0]} parts × {df.shape[1]} columns")

# Join CV image index
cv_idx = build_cv_index()
df     = join_cv_to_parquet(df, cv_idx)
print(f"After CV join : {df.shape}")

tabular_cols = get_tabular_cols(df)
dxp_cols     = get_dxp_cols(df)
print(f"\nTabular features : {len(tabular_cols)}")
print(f"DXP channels     : {len(dxp_cols)}")
print(f"Label columns    : {C.N_LABELS}")

In [ ]:
# ── Dataset summary ──────────────────────────────────────────
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(df.dtypes.value_counts().to_string())
print(f"\nMissing values (top-10):")
miss = df.isnull().sum().sort_values(ascending=False)
print(miss[miss > 0].head(10).to_string())
print(f"\nExperiment groups : {df[C.GROUP_COL].nunique()}")
print(f"Parts per group   : {df.groupby(C.GROUP_COL).size().describe().to_string()}")

## 2 · Label Distribution — Class Imbalance

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(C.LABEL_COLS):
    counts = df[col].value_counts()
    nok_pct = counts.get(1, 0) / len(df) * 100
    colour  = "#E53935" if nok_pct > 20 else "#FB8C00" if nok_pct > 10 else "#43A047"
    axes[i].bar(["OK (0)", "NOK (1)"], [counts.get(0,0), counts.get(1,0)],
                color=["#78909C", colour], edgecolor="white", linewidth=0.8)
    axes[i].set_title(col.replace("LBL_",""), fontsize=10)
    axes[i].set_ylabel("Parts")
    axes[i].text(1, counts.get(1,0) + 1, f"{nok_pct:.1f}%", ha="center",
                 fontsize=9, color=colour, fontweight="bold")

fig.suptitle("Label Distribution — Positive (NOK) Rate per Defect Type", fontsize=13)
plt.tight_layout()
plt.show()
print("Most labels are severely imbalanced → Focal Loss required.")

## 3 · The Leakage Problem
> **Key Research Finding**: XGBoost and BiLSTM achieve **F1 = 1.00** on random splits
> but only **F1 = 0.19–0.47** under group-aware k-fold.
> Root cause: the Taguchi DOE design means the 47 experiment cells each have
> a *fixed setpoint signature* → models memorise SET_ToolTemperature → defect,
> not the actual physics. Our 4-modal system with DANN breaks this.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, GroupKFold
from sklearn.preprocessing import StandardScaler

tab_cols = [c for c in tabular_cols if c in df.columns]
X_raw    = df[tab_cols].fillna(df[tab_cols].median())
y        = df[C.PRIMARY_TARGET].fillna(0).astype(int).values
groups   = df[C.GROUP_COL].values

scaler   = StandardScaler()
X        = scaler.fit_transform(X_raw)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

# Random 10-fold (the misleading result)
skf        = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
random_f1s = cross_val_score(clf, X, y, cv=skf, scoring="f1", n_jobs=-1)

# Group k-fold (the honest result)
gkf        = GroupKFold(n_splits=5)
group_f1s  = cross_val_score(clf, X, y, cv=gkf, groups=groups, scoring="f1", n_jobs=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.boxplot([random_f1s, group_f1s], labels=["Random 10-fold\n(MISLEADING)","Group 5-fold\n(HONEST)"],
            patch_artist=True,
            boxprops=dict(facecolor="#EF9A9A"),
            medianprops=dict(color="#B71C1C", linewidth=2.5))
ax1.set_ylabel("F1 Score (LBL_NOK)")
ax1.set_title("Tabular-Only: Random vs Group CV")
ax1.set_ylim(0, 1.05)
ax1.axhline(random_f1s.mean(), ls="--", color="#C62828", alpha=0.5)
ax1.axhline(group_f1s.mean(),  ls="--", color="#1565C0", alpha=0.5)

# Gap bar
gap = random_f1s.mean() - group_f1s.mean()
ax2.bar(["Random F1", "Group F1", "Leakage Gap"],
        [random_f1s.mean(), group_f1s.mean(), gap],
        color=["#EF5350","#42A5F5","#FF7043"])
ax2.set_title(f"Leakage = {gap:.3f} F1 points")
ax2.set_ylim(0, 1.1)
for rect, val in zip(ax2.patches, [random_f1s.mean(), group_f1s.mean(), gap]):
    ax2.text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.02,
             f"{val:.3f}", ha="center", fontweight="bold")

plt.suptitle("The Leakage Problem — Why 4-Modal DANN is Needed", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Random  F1 : {random_f1s.mean():.4f} ± {random_f1s.std():.4f}  ← inflated by setpoint memorisation")
print(f"Group   F1 : {group_f1s.mean():.4f}  ± {group_f1s.std():.4f}  ← true generalisation")
print(f"Leakage gap: {gap:.4f}  ← our DANN + thermal/CV modalities must close this")

## 4 · Thermal Imagery — Z01-Diff Analysis

In [ ]:
from data.loaders import load_thermal_pair, extract_thermal_rois

# Sample a part that has thermal data
thermal_rows = df[df["has_thermal"]].head(3) if "has_thermal" in df.columns else df.head(3)

if len(thermal_rows) == 0:
    print("No thermal CSV files found at", C.THERMAL_CSV_DIR)
    print("(Expected when running from local system without the full dataset)")
else:
    fig, axes = plt.subplots(len(thermal_rows), 3, figsize=(15, 4 * len(thermal_rows)))
    if len(thermal_rows) == 1:
        axes = axes[np.newaxis, :]

    cmaps = ["inferno", "inferno", "RdBu_r"]
    titles = ["Frame 1 (filling)", "Frame 2 (cooling)", "Z01-Diff (F1–F2)"]

    for row_i, (_, row) in enumerate(thermal_rows.iterrows()):
        try:
            imgs = load_thermal_pair(row)        # (3, 480, 640)
            for ch, (cmap, title) in enumerate(zip(cmaps, titles)):
                im = axes[row_i, ch].imshow(imgs[ch], cmap=cmap, interpolation="lanczos")
                axes[row_i, ch].set_title(f"Part {row.get(C.CYCLE_ID_COL,'?')} — {title}")
                axes[row_i, ch].axis("off")
                plt.colorbar(im, ax=axes[row_i, ch], fraction=0.046, pad=0.04)
        except Exception as e:
            axes[row_i, 0].text(0.5, 0.5, f"Load error:\n{e}",
                                ha="center", va="center", transform=axes[row_i, 0].transAxes)

    plt.suptitle("Thermal Image Triplet: [Frame1 | Frame2 | Diff] per Part", fontsize=13)
    plt.tight_layout()
    plt.show()

## 5 · Visual Imagery — 3-Section Warpage Shell

In [ ]:
cv_rows = df[df.get("cv_path_pos1","").notna()].head(2) if "cv_path_pos1" in df.columns else pd.DataFrame()

if cv_rows.empty:
    print("No CV images joined. Check CV_IMAGE_DIR in config.py")
    print("Expected path:", C.CV_IMAGE_DIR)
else:
    from data.loaders import load_cv_triple
    fig, axes = plt.subplots(len(cv_rows), 3, figsize=(15, 5 * len(cv_rows)))
    if len(cv_rows) == 1:
        axes = axes[np.newaxis, :]
    pos_labels = ["Left section", "Mid section", "Right section"]

    for ri, (_, row) in enumerate(cv_rows.iterrows()):
        try:
            imgs = load_cv_triple(row)     # (3, H, W) float32
            for ci in range(3):
                axes[ri, ci].imshow(imgs[ci], cmap="gray", vmin=0, vmax=1)
                axes[ri, ci].set_title(f"Part {row.get(C.CYCLE_ID_COL,'?')} — {pos_labels[ci]}")
                axes[ri, ci].axis("off")
        except Exception as e:
            axes[ri, 0].text(0.5, 0.5, str(e), ha="center", transform=axes[ri, 0].transAxes)

    plt.suptitle("CV Images: Left / Mid / Right sections of Warpage Shell", fontsize=13)
    plt.tight_layout()
    plt.show()

## 6 · DXP Process Signals — Injection Cycle Dynamics

In [ ]:
from data.loaders import load_dxp_tensor

# Pick a NOK and OK part for comparison
nok_rows = df[df[C.PRIMARY_TARGET] == 1].head(1)
ok_rows  = df[df[C.PRIMARY_TARGET] == 0].head(1)
sample_rows = pd.concat([ok_rows, nok_rows]).reset_index(drop=True)
labels_plot  = ["OK", "NOK"]

dxp_available = [c for c in C.KEY_DXP_CHANNELS if c in df.columns]
if not dxp_available:
    print("DXP columns not found. (Expected when dataset is not loaded locally)")
else:
    n_ch = len(dxp_available)
    fig, axes = plt.subplots(n_ch, 1, figsize=(14, 2.2 * n_ch), sharex=True)
    colours = ["#1976D2", "#D32F2F"]

    for ri, (_, row) in enumerate(sample_rows.iterrows()):
        try:
            seq = load_dxp_tensor(row, dxp_available)   # (n_ch, T)
            for ci in range(n_ch):
                x = np.linspace(0, 1, seq.shape[1])
                axes[ci].plot(x, seq[ci], color=colours[ri], alpha=0.75,
                              linewidth=1.2, label=labels_plot[ri])
                axes[ci].set_ylabel(dxp_available[ci].replace("DXP_",""), fontsize=8)
                axes[ci].tick_params(labelsize=7)
        except Exception as e:
            axes[0].text(0.5, 0.5, str(e), ha="center", transform=axes[0].transAxes)

    axes[-1].set_xlabel("Normalised Cycle Time")
    axes[0].legend(loc="upper right")
    fig.suptitle("DXP Key Channels: OK vs NOK Injection Cycle", fontsize=13)
    plt.tight_layout()
    plt.show()

## 7 · Taguchi DOE Cell Structure

In [ ]:
# Visualise how parts are distributed across experiment cells
grp_sizes  = df.groupby(C.GROUP_COL).size().sort_values(ascending=False)
grp_nok    = df.groupby(C.GROUP_COL)[C.PRIMARY_TARGET].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Parts per cell
bars = ax1.bar(range(len(grp_sizes)), grp_sizes.values,
               color=sns.color_palette("Blues_d", len(grp_sizes)))
ax1.set_title(f"Parts per Experiment Cell (total={df[C.GROUP_COL].nunique()} cells)")
ax1.set_xlabel("Experiment Cell (sorted by count)")
ax1.set_ylabel("Number of Parts")
ax1.axhline(grp_sizes.mean(), ls="--", color="red", label=f"Mean={grp_sizes.mean():.1f}")
ax1.legend()

# NOK rate per cell — colour-coded
nok_sorted = grp_nok.sort_values(ascending=False)
bar_colours = ["#EF5350" if v > 0.5 else "#FF9800" if v > 0.2 else "#66BB6A"
               for v in nok_sorted.values]
ax2.bar(range(len(nok_sorted)), nok_sorted.values, color=bar_colours)
ax2.axhline(0.2,  ls="--", color="orange",  label="20% threshold (YELLOW)")
ax2.axhline(0.5,  ls="--", color="red",     label="50% threshold (RED)")
ax2.set_title("NOK Rate per Experiment Cell (Taguchi DOE)")
ax2.set_xlabel("Experiment Cell (sorted by NOK rate)")
ax2.set_ylabel("NOK Rate")
ax2.set_ylim(0, 1.05)
ax2.legend()

fig.suptitle("Taguchi DOE: 47 Cells × ~12 Parts — Group-Aware Split is Mandatory", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Cells with NOK rate > 50%: {(grp_nok > 0.5).sum()}")
print(f"Cells with NOK rate < 20%: {(grp_nok < 0.2).sum()}")
print(f"\nConclusion: models must generalise to UNSEEN experiment cells,")
print(f"not memorise which setpoints cause defects.")

## 8 · Key Findings

In [ ]:
findings = {
    "Dataset"        : f"{len(df)} parts, {df.shape[1]} columns, 9 data sources",
    "Labels"         : f"{C.N_LABELS} multi-label targets, severely imbalanced",
    "Leakage gap"    : f"{gap:.3f} F1 — tabular-only model memorises Taguchi setpoints",
    "Thermal"        : f"{df['has_thermal'].sum() if 'has_thermal' in df.columns else 'N/A'} parts with IR CSVs (480×640, 2 frames)",
    "CV images"      : f"3 sections per part, 2448×2048 BMP greyscale",
    "DXP channels"   : f"{len(dxp_cols)} available, {len(C.KEY_DXP_CHANNELS)} key channels selected",
    "Experiments"    : f"{df[C.GROUP_COL].nunique()} Taguchi cells, GroupKFold required",
    "Solution"       : "4-Modal DANN Fusion — thermal+CV encode physics, not setpoints",
}

print("=" * 60)
print("EDA KEY FINDINGS")
print("=" * 60)
for k, v in findings.items():
    print(f"  {k:16s} : {v}")
print("=" * 60)